In [1]:
import pandas as pd

print("Loading NQ_merged.csv")

a = pd.read_csv("NQ_merged.csv")

a["datetime"] = pd.to_datetime(
    a["time"],
    unit="s",
    utc=True
)

a = a.rename(
    columns={"Volume": "volume"}
)

a = a[
    [
        "datetime",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

print("Rows:", len(a))

print()
print("Loading databento_continuous.csv")

b = pd.read_csv("databento_continuous.csv")

b["datetime"] = pd.to_datetime(
    b["datetime"],
    utc=True
)

b = b[
    [
        "datetime",
        "open",
        "high",
        "low",
        "close",
        "volume"
    ]
]

print("Rows:", len(b))

# Overlap period
start_overlap = max(
    a["datetime"].min(),
    b["datetime"].min()
)

end_overlap = min(
    a["datetime"].max(),
    b["datetime"].max()
)

print()
print("Overlap Start:", start_overlap)
print("Overlap End  :", end_overlap)

a = a[
    (a["datetime"] >= start_overlap)
    &
    (a["datetime"] <= end_overlap)
]

b = b[
    (b["datetime"] >= start_overlap)
    &
    (b["datetime"] <= end_overlap)
]

merged = pd.merge(
    a,
    b,
    on="datetime",
    how="inner",
    suffixes=("_nq", "_db")
)

print()
print("Matched timestamps:", len(merged))

differences = merged[
    (merged["open_nq"] != merged["open_db"])
    |
    (merged["high_nq"] != merged["high_db"])
    |
    (merged["low_nq"] != merged["low_db"])
    |
    (merged["close_nq"] != merged["close_db"])
    |
    (merged["volume_nq"] != merged["volume_db"])
]

matches = len(merged) - len(differences)

print()
print("Matching candles :", matches)
print("Different candles:", len(differences))

if len(merged):
    print(
        "Match %:",
        round(matches * 100 / len(merged), 6)
    )

differences.to_csv(
    "comparison_differences.csv",
    index=False
)

print()
print(
    "Differences saved to comparison_differences.csv"
)

if len(differences):
    print()
    print("First mismatches:")
    print(differences.head(20))

Loading NQ_merged.csv
Rows: 352952

Loading databento_continuous.csv
Rows: 352957

Overlap Start: 2025-02-21 00:00:00+00:00
Overlap End  : 2026-02-20 21:59:00+00:00

Matched timestamps: 352952

Matching candles : 349952
Different candles: 3000
Match %: 99.150026

Differences saved to comparison_differences.csv

First mismatches:
                      datetime   open_nq   high_nq    low_nq  close_nq  \
870  2025-02-21 14:30:00+00:00  22186.25  22188.00  22158.00  22160.00   
1169 2025-02-21 19:29:00+00:00  21705.00  21708.25  21670.50  21679.50   
1257 2025-02-21 20:57:00+00:00  21683.00  21688.50  21675.00  21686.25   
2823 2025-02-25 01:04:00+00:00  21435.00  21448.50  21433.75  21446.25   
3025 2025-02-25 04:26:00+00:00  21436.25  21437.75  21436.25  21437.50   
4206 2025-02-26 01:07:00+00:00  21234.00  21236.50  21232.00  21232.75   
4934 2025-02-26 13:15:00+00:00  21298.75  21302.25  21295.50  21298.00   
5009 2025-02-26 14:30:00+00:00  21238.75  21240.75  21192.75  21193.25   
501

In [3]:
import pandas as pd
test = pd.read_csv("comparison_differences.csv")
test

,datetime,open_nq,high_nq,low_nq,close_nq,volume_nq,open_db,high_db,low_db,close_db,volume_db
0,2025-02-21 14:30:00+00:00,22186.25,22188.00,22158.00,22160.00,5866,22186.25,22188.00,22158.00,22160.00,5881
1,2025-02-21 19:29:00+00:00,21705.00,21708.25,21670.50,21679.50,4697,21705.00,21708.25,21670.50,21679.50,4721
2,2025-02-21 20:57:00+00:00,21683.00,21688.50,21675.00,21686.25,2654,21683.00,21688.50,21675.00,21686.25,2666
3,2025-02-25 01:04:00+00:00,21435.00,21448.50,21433.75,21446.25,251,21435.00,21448.50,21433.75,21446.25,252
4,2025-02-25 04:26:00+00:00,21436.25,21437.75,21436.25,21437.50,9,21673.00,21675.00,21673.00,21675.00,10
...,...,...,...,...,...,...,...,...,...,...,...
2995,2026-02-04 21:35:00+00:00,25171.50,25181.25,25163.50,25163.50,879,25171.50,25181.25,25163.50,25163.50,880
2996,2026-02-10 16:15:00+00:00,25376.00,25383.50,25349.00,25374.75,1834,25376.00,25383.50,25349.00,25374.75,1837
2997,2026-02-17 14:39:00+00:00,24777.00,24815.50,24772.50,24798.00,2041,24777.00,24815.75,24772.50,24798.00,2326
2998,2026-02-17 20:59:00+00:00,24759.25,24782.25,24744.25,24776.75,7533,24759.25,24782.25,24744.25,24777.00,7531
